# SEISMICPIPELINE: PREDICCION DE RIESGO DE TSUNAMI MEDIANTE XGBOOST

<br>

**Institucion:** Universidad Internacional del Ecuador (UIDE)

**Escuela:** Ciencias de la Computacion

**Asignaturas:** Big Data - Machine Learning - Gestion de Proyectos de SI - Ciberseguridad

**Semestre:** Sexto - 2026

**Jira:** Proyecto SEIS

**Repositorio:** github.com/DanielSozoranga/tsunami-risk-pipeline

<br>

**Equipo Scrum:**

| Integrante | Roles |
|---|---|
| **Daniel Sozoranga** | Scrum Master · Development Team |
| **Ricardo Álvarez** | Product Owner · Development Team |

<br>

---

## PREGUNTA QUE RESPONDE EL MODELO

**?Este sismo generara tsunami?**

El modelo es un clasificador binario **XGBoost** que, dadas las caracteristicas fisicas de un sismo, devuelve una **probabilidad continua entre 0 y 1** (`predict_proba`) de que el evento genere tsunami. Esa probabilidad se proyecta sobre el registro sismico costero ecuatoriano como un **score de riesgo por provincia** (Esmeraldas, Manabi, Santa Elena, Guayas, El Oro, Galapagos), no como clasificacion binaria.

<br>

---

## ESTRUCTURA DEL NOTEBOOK

| Seccion | Contenido |
|---|---|
| **Seccion 1** | Instalacion y Configuracion del Entorno |
| **Seccion 2** | Importacion de Librerias |
| **Seccion 3** | Conexion y Extraccion de Datos (API USGS) |
| **Seccion 4** | Variables Obtenidas - Dataset Original |
| **Seccion 5** | Renombrado de Variables: API -> Espanol |
| **Seccion 6** | Limpieza y Preprocesamiento (ETL) |
| **Seccion 7** | Analisis Exploratorio (EDA) |
| **Seccion 8** | Benchmarking de Engines: Pandas vs PySpark vs Dask |
| **Seccion 9** | Preparacion del Dataset de Machine Learning |
| **Seccion 10** | Entrenamiento del Clasificador XGBoost |
| **Seccion 11** | Diagnostico de Entrenamiento |
| **Seccion 12** | Evaluacion del Modelo |
| **Seccion 13** | Modelos Baseline de Comparacion |
| **Seccion 14** | Justificacion Tecnica de la Seleccion de XGBoost |
| **Seccion 15** | Proyeccion de Riesgo sobre Ecuador |
| **Seccion 16** | Validacion Internacional (Tohoku 2011 / Maule 2010) |
| **Seccion 17** | Dashboard Interactivo de Resultados |
| **Seccion 18** | Analisis Critico: Limitaciones y Mejoras |
| **Seccion 19** | Conclusiones |

<br>

---

## CONVENCIONES

- **Semilla aleatoria global:** `SEED = 42` - reutilizada en todo split y modelo para garantizar reproducibilidad.
- **Idioma de las variables:** los datos llegan de la API con nombres en ingles. En la Seccion 5 se renombran a espanol con tabla de equivalencias original -> final.
- **Features del modelo (7):** `magnitud`, `profundidad_km`, `latitud`, `longitud`, `significancia`, `num_estaciones`, `brecha_azimutal`. La seleccion se justifica en la Seccion 7 con la matriz de correlacion completa.
- **Target:** `tsunami` (0 = no genero tsunami, 1 = si genero tsunami), bandera oficial del catalogo USGS.
- **Modelo unico:** XGBoost. Random Forest y Regresion Logistica aparecen solo como baselines de comparacion (Seccion 13).

<br>

---

# SECCION 1 - INSTALACION Y CONFIGURACION DEL ENTORNO

<br>

Se instalan las librerias que no estan disponibles por defecto en Google Colab.

| Libreria | Proposito |
|---|---|
| **xgboost** | Algoritmo principal del proyecto (clasificador de tsunamis) |
| **pyspark** | Procesamiento distribuido (motor Big Data del benchmarking) |
| **dask** | Procesamiento paralelo con ejecucion lazy (benchmarking) |
| **plotly** | Dashboard interactivo de resultados |
| **psutil** | Medicion de consumo de memoria RAM en tiempo real |
| **joblib** | Serializacion del modelo entrenado |

<br>

In [ ]:
# Librerias que Google Colab no tiene preinstaladas.
# Si ejecutas localmente, asegurate de tener Python 3.10+
# y un entorno virtual activo antes de correr este comando.
!pip install xgboost pyspark "dask[dataframe]" plotly psutil joblib --quiet

print("Librerias instaladas correctamente")


---

# SECCION 2 - IMPORTACION DE LIBRERIAS

<br>

In [ ]:
# --- Consumo de API y utilidades ---
import os
import time
import math
import json
import warnings
import requests

# --- Manipulacion de datos ---
import numpy  as np
import pandas as pd

# --- Visualizacion estatica ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Visualizacion interactiva ---
import plotly.graph_objects as go
import plotly.express       as px
from   plotly.subplots      import make_subplots

# --- Machine Learning ---
from sklearn.model_selection  import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.metrics          import (roc_auc_score, roc_curve,
                                      confusion_matrix, classification_report,
                                      precision_score, recall_score, f1_score,
                                      ConfusionMatrixDisplay)
from sklearn.ensemble         import RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import StandardScaler
from xgboost                  import XGBClassifier
import xgboost as xgb

# --- Persistencia y metricas del sistema ---
import joblib   # Serializacion del modelo entrenado
import psutil   # Medicion de consumo de RAM en tiempo real

# --- Configuracion global de visualizacion ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)

# Semilla global: garantiza reproducibilidad en splits, modelos y muestras
SEED = 42
np.random.seed(SEED)

print("Librerias importadas correctamente")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  xgboost : {xgb.__version__}")
print(f"  SEED    : {SEED}")


---

# SECCION 3 - CONEXION Y EXTRACCION DE DATOS (API USGS)

<br>

## 3.1 Descripcion de la Fuente

<br>

| Parametro | Detalle |
|---|---|
| **Endpoint** | https://earthquake.usgs.gov/fdsnws/event/1/query |
| **Autenticacion** | Ninguna (API 100% publica) |
| **Estandar** | FDSN Event Web Service Specification |
| **Formato de respuesta** | GeoJSON (FeatureCollection) |
| **Cobertura temporal** | 1900 - presente |
| **Cobertura geografica** | Global |

<br>

## 3.2 Parametros de Consulta

<br>

Se extrae el **Cinturon de Fuego del Pacifico (1990-2024)** con los siguientes parametros:

| Parametro | Valor | Justificacion |
|---|---|---|
| `minmagnitude` | 5.0 | Eventos con potencial tsunamigenico real |
| `minlatitude` / `maxlatitude` | -60 / 65 | Desde la placa Antartica hasta Alaska/Kamchatka |
| `minlongitude` / `maxlongitude` | 110 / 300 | Cuenca del Pacifico (300 = -60W, permite cruzar el antimeridiano) |
| `limit` | 20000 | Limite maximo de la API por request -> requiere paginacion anual |

La paginacion anual (1990-2024 = 35 requests) evita superar el limite de la API. El retry con backoff exponencial protege contra fallas transitorias de la red.

<br>

## 3.3 Configuracion del Entorno de Archivos

<br>

In [ ]:
# =============================================================
#  SECCION 3.3 - CONFIGURACION DE CARPETAS
# =============================================================
#  Estructura estandar del proyecto:
#    /data/raw        <- datasets crudos directamente de la API
#    /data/processed  <- datasets limpios y artefactos del modelo
#    /notebooks       <- notebooks de analisis y exploracion
#    /models          <- modelos serializados (.pkl)
# =============================================================

try:
    # Entorno Google Colab: monta Drive automaticamente
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/SeismicPipeline'
except ModuleNotFoundError:
    # Entorno local: crea las carpetas relativas al directorio actual
    BASE = os.path.abspath('./SeismicPipeline')

DIRS = {
    'raw'       : f'{BASE}/data/raw',
    'processed' : f'{BASE}/data/processed',
    'notebooks' : f'{BASE}/notebooks',
    'models'    : f'{BASE}/models',
}

for nombre, ruta in DIRS.items():
    os.makedirs(ruta, exist_ok=True)   # exist_ok: no falla si ya existe
    print(f"  [OK] {nombre:10s} -> {ruta}")

print()
print("Estructura de carpetas configurada correctamente")


<br>

## 3.4 Funciones de Extraccion

<br>

In [ ]:
# =============================================================
#  SECCION 3.4 - FUNCIONES DE EXTRACCION
# =============================================================

# URL base del servicio FDSN (Federal Digital Seismograph Network)
BASE_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query'

# Bounding box del Cinturon de Fuego del Pacifico.
# La API acepta longitudes > 180 para cruzar el antimeridiano
# (ej. 300 = 360 - 60 = 60 W), lo que permite cubrir el Pacifico
# sin necesidad de dividir la consulta.
BBOX_RING_OF_FIRE = {
    'minlatitude'  : -60,   # Desde la placa Antartica
    'maxlatitude'  :  65,   # Hasta Alaska / Kamchatka
    'minlongitude' : 110,   # Extremo occidental: Filipinas
    'maxlongitude' : 300,   # Extremo oriental: Costa Rica (via antimeridiano)
}

MIN_MAGNITUD = 5.0   # Umbral minimo: eventos con potencial tsunamigenico real
ANIO_INICIO  = 1990
ANIO_FIN     = 2024


def consultar_usgs(params: dict, timeout: int = 60) -> dict:
    '''
    Realiza un GET a la API USGS y retorna el JSON decodificado.

    Parametros:
        params  : dict - parametros de la consulta HTTP
        timeout : int  - segundos maximos de espera (default 60)

    Retorna:
        dict con la respuesta GeoJSON de la API.

    Lanza:
        RuntimeError si la respuesta es un error HTTP, timeout o falla de red.
    '''
    try:
        resp = requests.get(BASE_URL, params=params, timeout=timeout)

        # Errores del cliente (400-499): problema en los parametros enviados
        if 400 <= resp.status_code < 500:
            raise RuntimeError(f"Error de cliente {resp.status_code}: {resp.text[:200]}")

        # Errores del servidor (500+): USGS no disponible temporalmente
        if resp.status_code >= 500:
            raise RuntimeError(f"Error de servidor {resp.status_code}: USGS no disponible")

        data = resp.json()

        # La API USGS siempre incluye 'features' en una respuesta valida
        if 'features' not in data:
            raise RuntimeError(f"Respuesta inesperada: {str(data)[:200]}")

        return data

    except requests.exceptions.Timeout:
        raise RuntimeError(f"Timeout: la API no respondio en {timeout}s")
    except requests.exceptions.ConnectionError as e:
        raise RuntimeError(f"Error de conexion: {e}")


def consultar_con_retry(params: dict, max_reintentos: int = 4) -> dict:
    '''
    Envuelve consultar_usgs() con reintentos de backoff exponencial.

    Parametros:
        params         : dict - parametros de la consulta
        max_reintentos : int  - numero maximo de reintentos (default 4)

    Estrategia de espera: 2^(intento+1) segundos -> 2, 4, 8, 16 seg.
    Esto evita saturar la API cuando hay fallas transitorias de red.
    '''
    for intento in range(max_reintentos + 1):
        try:
            return consultar_usgs(params)
        except RuntimeError as e:
            if intento == max_reintentos:
                raise   # Agotados los reintentos: propagar el error
            espera = 2 ** (intento + 1)
            print(f"  [retry {intento+1}/{max_reintentos}] {e} - esperando {espera}s...")
            time.sleep(espera)


# Prueba de conexion con 3 eventos recientes para verificar que la API responde
prueba   = consultar_usgs({'format': 'geojson', 'minmagnitude': 6, 'limit': 3, 'orderby': 'time'})
n_prueba = prueba.get('metadata', {}).get('count', len(prueba['features']))
print(f"Conexion a la API USGS: OK")
print(f"  Eventos recibidos en prueba: {n_prueba}")


<br>

## 3.5 Extraccion Masiva Ring of Fire 1990-2024

<br>

La extraccion trae **todas las propiedades disponibles de la API** con sus nombres originales en ingles. El renombrado a espanol se realiza en la Seccion 5, de forma trazable, para que el docente pueda comparar el dataset original con el dataset transformado.

<br>

In [ ]:
# =============================================================
#  SECCION 3.5 - EXTRACCION MASIVA CON PAGINACION ANUAL
# =============================================================
#  La API limita cada request a 20,000 eventos.
#  Para cubrir 1990-2024 (35 anios) se hacen 35 requests,
#  uno por anio. Total esperado: ~15,000 - 30,000 eventos.
# =============================================================

# Propiedades extraidas de la API - se conservan los NOMBRES ORIGINALES
# (el renombrado a espanol ocurre en la Seccion 5)
PROPS_API = [
    'mag',     # Magnitud del sismo
    'sig',     # Indice de significancia USGS
    'nst',     # Numero de estaciones que reportaron
    'gap',     # Brecha azimutal (calidad de localizacion)
    'dmin',    # Distancia a la estacion mas cercana
    'rms',     # Error cuadratico medio del ajuste de tiempo
    'felt',    # Numero de reportes de personas que lo sintieron
    'cdi',     # Intensidad maxima comunitaria (CDI)
    'mmi',     # Intensidad maxima instrumental (MMI)
    'alert',   # Nivel de alerta PAGER
    'magType', # Tipo de magnitud (Mw, mb, ml, etc.)
    'tsunami', # Bandera oficial: 1 si el evento genero tsunami
    'place',   # Descripcion textual del lugar
    'time',    # Timestamp en milisegundos UTC
]


def extraer_ring_of_fire() -> pd.DataFrame:
    '''
    Extrae el catalogo sismico del Cinturon de Fuego del Pacifico (1990-2024).

    Realiza 35 requests anuales a la API USGS paginando por anio para respetar
    el limite de 20,000 eventos por consulta. Incluye un log de extraccion por anio.

    Retorna:
        pd.DataFrame con todos los eventos y sus propiedades originales de la API.
    '''
    registros = []   # Acumula cada evento como diccionario
    log       = []   # Registro de cuantos eventos se obtuvieron por anio

    for anio in range(ANIO_INICIO, ANIO_FIN + 1):
        params = {
            'format'       : 'geojson',
            'starttime'    : f'{anio}-01-01',
            'endtime'      : f'{anio}-12-31T23:59:59',
            'minmagnitude' : MIN_MAGNITUD,
            'limit'        : 20000,          # Maximo permitido por la API
            **BBOX_RING_OF_FIRE,
        }
        data  = consultar_con_retry(params)
        feats = data.get('features', [])

        for f in feats:
            p, g = f['properties'], f['geometry']['coordinates']
            # La geometria GeoJSON devuelve [longitud, latitud, profundidad]
            fila = {
                'id'        : f['id'],
                'longitude' : g[0],   # Longitud del epicentro
                'latitude'  : g[1],   # Latitud del epicentro
                'depth'     : g[2],   # Profundidad del hipocentro en km
            }
            for prop in PROPS_API:
                fila[prop] = p.get(prop)   # None si la API no reporta la propiedad
            registros.append(fila)

        log.append({'anio': anio, 'eventos': len(feats)})
        print(f"  {anio}: {len(feats):5,d} eventos extraidos")
        time.sleep(0.5)   # Pausa de cortesia para no saturar la API (rate limiting)

    df = pd.DataFrame(registros)
    pd.DataFrame(log).to_csv(f"{DIRS['raw']}/log_extraccion.csv", index=False)
    return df


# Estrategia de carga: reutilizar el dataset si ya existe en Drive.
# Si la version guardada no tiene todas las columnas (version antigua),
# se re-extrae automaticamente para garantizar integridad.
RUTA_RAW = f"{DIRS['raw']}/usgs_raw.csv"

if os.path.exists(RUTA_RAW):
    df_raw = pd.read_csv(RUTA_RAW)
    if 'dmin' not in df_raw.columns:
        print("Version anterior detectada (columnas incompletas). Re-extrayendo...")
        df_raw = extraer_ring_of_fire()
        df_raw.to_csv(RUTA_RAW, index=False)
    else:
        print("Dataset crudo encontrado en Drive. Cargando sin re-extraer...")
else:
    print("Primera ejecucion: iniciando extraccion masiva 1990-2024 (~2 minutos)...")
    df_raw = extraer_ring_of_fire()
    df_raw.to_csv(RUTA_RAW, index=False)

print()
print("=" * 58)
print("  RESUMEN DE EXTRACCION - API USGS Earthquake Catalog")
print("=" * 58)
print(f"  Filas       : {df_raw.shape[0]:>10,d} registros")
print(f"  Columnas    : {df_raw.shape[1]:>10,d} variables")
print(f"  Periodo     :       1990 - 2024 (Ring of Fire)")
print(f"  Magnitud    :       M >= {MIN_MAGNITUD}")
print(f"  Exportado a : {RUTA_RAW}")
print("=" * 58)


---

# SECCION 4 - VARIABLES OBTENIDAS - DATASET ORIGINAL

<br>

Esta seccion muestra el dataset **tal como llega de la API USGS**, con sus nombres originales en ingles, antes de cualquier transformacion. Esto permite comparar el dataset original con el dataset transformado en la Seccion 5.

<br>

## 4.1 Catalogo de Variables de la API

<br>

Todas las propiedades que el endpoint GeoJSON de la USGS puede entregar para cada evento sismico:

| Variable (API) | Tipo | Descripcion |
|---|---|---|
| `id` | str | Identificador unico del evento |
| `longitude` | float | Longitud del epicentro (geometry[0]) |
| `latitude` | float | Latitud del epicentro (geometry[1]) |
| `depth` | float | Profundidad del hipocentro en km (geometry[2]) |
| `mag` | float | Magnitud del sismo |
| `magType` | str | Tipo de escala de magnitud (mww, mb, ml...) |
| `sig` | int | Indice de significancia USGS (0-1000+) |
| `nst` | int | Numero de estaciones sismicas utilizadas |
| `gap` | float | Brecha azimutal maxima en grados |
| `dmin` | float | Distancia a la estacion mas cercana (grados) |
| `rms` | float | Error RMS del ajuste temporal de la solucion |
| `felt` | float | Numero de reportes publicos (Did You Feel It?) |
| `cdi` | float | Intensidad maxima percibida por la comunidad |
| `mmi` | float | Intensidad instrumental maxima (ShakeMap) |
| `alert` | str | Nivel de alerta PAGER (green/yellow/orange/red) |
| `tsunami` | int | **TARGET**: indica si el evento genero tsunami (0/1) |
| `place` | str | Descripcion textual de la ubicacion |
| `time` | int | Timestamp Unix del evento (milisegundos) |

<br>

## 4.2 Vista del Dataset Original

<br>

In [ ]:
# =============================================================
#  SECCION 4.2 - VISTA DEL DATASET ORIGINAL (nombres API)
# =============================================================

print("Dataset original - primeras 5 filas (nombres originales de la API USGS):")
print()
display(df_raw.head())

print()
print("Columnas originales de la API:")
print(list(df_raw.columns))

<br>

## 4.3 Estructura y Tipos de Datos

<br>

In [ ]:
# =============================================================
#  SECCION 4.3 - ESTRUCTURA DEL DATASET ORIGINAL
# =============================================================

print("Estructura del dataset crudo:")
print()
df_raw.info()

print()
print("=" * 58)
print("  RESUMEN DE VALORES NULOS - Dataset Original")
print("=" * 58)
nulos = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
nulos = nulos[nulos > 0].sort_values(ascending=False)
for var, pct in nulos.items():
    n = df_raw[var].isnull().sum()
    print(f"  {var:12s} : {n:6,d} nulos  ({pct:5.1f}%)")
print("=" * 58)

---

# SECCION 5 - RENOMBRADO DE VARIABLES: API -> ESPANOL

<br>

Para que el pipeline sea legible y evaluable, todas las columnas se renombran de sus abreviaturas tecnicas en ingles a **nombres descriptivos en espanol**. La siguiente tabla es la evidencia de trazabilidad original -> final que el docente puede verificar.

<br>

## 5.1 Tabla de Equivalencias

<br>

| Nombre Original (API USGS) | Nombre Final (Espanol) | Significado |
|---|---|---|
| `mag` | `magnitud` | Magnitud del sismo |
| `depth` | `profundidad_km` | Profundidad del hipocentro en kilometros |
| `latitude` | `latitud` | Latitud del epicentro |
| `longitude` | `longitud` | Longitud del epicentro |
| `sig` | `significancia` | Indice de significancia USGS (0-1000+) |
| `nst` | `num_estaciones` | Numero de estaciones sismicas utilizadas |
| `gap` | `brecha_azimutal` | Brecha azimutal maxima entre estaciones ( grados) |
| `dmin` | `dist_min_estacion` | Distancia a la estacion mas cercana ( grados) |
| `rms` | `error_rms` | Error RMS del ajuste temporal |
| `felt` | `reportes_sentido` | Numero de reportes ciudadanos |
| `cdi` | `intensidad_cdi` | Intensidad maxima percibida (Did You Feel It?) |
| `mmi` | `intensidad_mmi` | Intensidad instrumental Mercalli Modificada |
| `alert` | `nivel_alerta_pager` | Nivel de alerta PAGER |
| `magType` | `tipo_magnitud` | Metodo de calculo de la magnitud |
| `time` | `tiempo_ms` | Timestamp Unix en milisegundos |
| `place` | `lugar` | Descripcion textual de la ubicacion |
| `tsunami` | `tsunami` | **TARGET**: bandera 0/1 de tsunami asociado |

<br>

## 5.2 Aplicacion del Renombrado

<br>

In [ ]:
# =============================================================
#  SECCION 5.2 - RENOMBRADO DE COLUMNAS
# =============================================================
#  Los nombres originales de la API estan en ingles y son
#  abreviaciones opacas (mag, sig, nst, gap...).
#  Se mapean a nombres descriptivos en espanol para que todo
#  el pipeline posterior sea legible sin consultar la documentacion.
# =============================================================

MAPEO_COLUMNAS = {
    'mag'       : 'magnitud',
    'depth'     : 'profundidad_km',
    'latitude'  : 'latitud',
    'longitude' : 'longitud',
    'sig'       : 'significancia',
    'nst'       : 'num_estaciones',
    'gap'       : 'brecha_azimutal',
    'dmin'      : 'dist_min_estacion',
    'rms'       : 'error_rms',
    'felt'      : 'reportes_sentido',
    'cdi'       : 'intensidad_cdi',
    'mmi'       : 'intensidad_mmi',
    'alert'     : 'nivel_alerta_pager',
    'magType'   : 'tipo_magnitud',
    'time'      : 'tiempo_ms',
    'place'     : 'lugar',
}

print("ANTES del renombrado (columnas originales de la API USGS):")
print(list(df_raw.columns))
print()

# rename() con un diccionario es seguro: ignora silenciosamente
# las claves que no existen en el DataFrame
df_es = df_raw.rename(columns=MAPEO_COLUMNAS)

print("DESPUES del renombrado (columnas en espanol):")
print(list(df_es.columns))


In [ ]:
# =============================================================
#  SECCION 5.3 - EVIDENCIA LADO A LADO (antes vs despues)
# =============================================================

print("Dataset ORIGINAL (API USGS) - 3 filas:")
display(df_raw[['mag', 'depth', 'sig', 'nst', 'gap',
                'felt', 'cdi', 'mmi', 'tsunami']].head(3))

print()
print("Dataset TRANSFORMADO (Espanol) - mismas 3 filas:")
display(df_es[['magnitud', 'profundidad_km', 'significancia', 'num_estaciones',
               'brecha_azimutal', 'reportes_sentido', 'intensidad_cdi',
               'intensidad_mmi', 'tsunami']].head(3))

---

# SECCION 6 - LIMPIEZA Y PREPROCESAMIENTO (ETL)

<br>

El preprocesamiento sigue la fase de preparacion de datos del estandar CRISP-DM:

| Paso | Descripcion |
|---|---|
| 6.1 | Eliminacion de duplicados |
| 6.2 | Tratamiento de valores nulos |
| 6.3 | Normalizacion de tipos y rangos fisicos |
| 6.4 | Exportacion del dataset limpio |

<br>

## 6.1 Eliminacion de Duplicados

<br>

In [ ]:
# =============================================================
#  SECCION 6.1 - ELIMINACION DE DUPLICADOS
# =============================================================
#  Dos criterios de duplicado:
#  1. Mismo ID de evento USGS (el mas confiable)
#  2. Misma combinacion de tiempo + coordenadas + magnitud
#     (detecta eventos registrados con IDs distintos que son
#     fisicamente el mismo sismo)
# =============================================================

df_etl        = df_es.copy()
filas_inicial = len(df_etl)

print(f"ANTES de limpiar duplicados:")
print(f"  Filas: {filas_inicial:,}")
print()

# Criterio 1: duplicados por identificador unico del evento USGS
dup_id = df_etl['id'].duplicated().sum()
df_etl = df_etl.drop_duplicates(subset='id', keep='first')

# Criterio 2: duplicados exactos por posicion espacio-temporal
dup_exacto = df_etl.duplicated(
    subset=['tiempo_ms', 'latitud', 'longitud', 'magnitud']
).sum()
df_etl = df_etl.drop_duplicates(
    subset=['tiempo_ms', 'latitud', 'longitud', 'magnitud'], keep='first'
)

print(f"DESPUES de limpiar duplicados:")
print(f"  Filas: {len(df_etl):,}")
print()
print("=" * 50)
print("  DETALLE DE ELIMINACION")
print("=" * 50)
print(f"  Duplicados por id               : {dup_id}")
print(f"  Duplicados espacio-temporales   : {dup_exacto}")
print(f"  Total filas eliminadas          : {filas_inicial - len(df_etl):,}")
print("=" * 50)


<br>

## 6.2 Tratamiento de Valores Nulos

<br>

**Estrategia de imputacion (justificada con el EDA de la Seccion 7):**

| Variable | Estrategia | Justificacion |
|---|---|---|
| `magnitud`, `profundidad_km`, `latitud`, `longitud`, `tsunami` | Eliminar fila | Son criticas para el modelo; sus nulos son escasos y no imputables |
| `num_estaciones`, `brecha_azimutal`, `significancia` | Imputar mediana | Distribuciones sesgadas; eliminar perderia eventos tsunamigenicos escasos |

<br>

In [ ]:
# =============================================================
#  SECCION 6.2 - TRATAMIENTO DE VALORES NULOS
# =============================================================
#  Estrategia diferenciada segun el rol de cada columna:
#
#  CRITICAS (magnitud, profundidad, coordenadas, target):
#    -> Eliminar la fila. Sin estos datos el registro es inutilizable.
#
#  SECUNDARIAS (num_estaciones, brecha_azimutal, significancia):
#    -> Imputar por mediana. Son features de calidad de localizacion
#       que el modelo puede usar aunque el valor sea aproximado.
# =============================================================

COLS_CRITICAS = ['magnitud', 'profundidad_km', 'latitud', 'longitud', 'tsunami']
antes_nulos   = df_etl[COLS_CRITICAS].isnull().sum().sum()
antes_filas   = len(df_etl)

print(f"ANTES del tratamiento de nulos:")
print(f"  Filas                          : {antes_filas:,}")
print(f"  Nulos en columnas criticas     : {antes_nulos}")
print()

# Eliminar filas donde alguna columna critica es nula
df_etl = df_etl.dropna(subset=COLS_CRITICAS)
print(f"  Filas eliminadas por nulos criticos: {antes_filas - len(df_etl)}")
print()

# Imputar columnas secundarias por mediana (robusta a outliers)
for col in ['num_estaciones', 'brecha_azimutal', 'significancia']:
    n_nulos = df_etl[col].isna().sum()
    if n_nulos:
        mediana     = df_etl[col].median()
        df_etl[col] = df_etl[col].fillna(mediana)
        print(f"  Imputados {n_nulos:,} nulos en '{col}' con mediana = {mediana:.2f}")

print()
print(f"DESPUES del tratamiento de nulos:")
print(f"  Filas                          : {len(df_etl):,}")
print(f"  Nulos en features + target     : "
      f"{int(df_etl[['magnitud','profundidad_km','latitud','longitud','significancia','num_estaciones','brecha_azimutal','tsunami']].isna().sum().sum())}")


<br>

## 6.3 Normalizacion de Tipos y Rangos Fisicos

<br>

In [ ]:
# =============================================================
#  SECCION 6.3 - NORMALIZACION DE TIPOS Y RANGOS FISICOS
# =============================================================
#  Tres operaciones:
#  1. Forzar tipos de datos correctos (float64, int8)
#  2. Derivar columnas de fecha desde el timestamp en ms
#  3. Corregir longitudes > 180 y descartar valores imposibles
# =============================================================

print("ANTES - tipos de datos en las features:")
print(df_etl[['magnitud','profundidad_km','latitud','longitud',
              'significancia','num_estaciones','brecha_azimutal','tsunami']].dtypes.to_string())
print()

# 1. Conversion de tipos
df_etl = df_etl.astype({
    'magnitud'        : 'float64',
    'profundidad_km'  : 'float64',
    'latitud'         : 'float64',
    'longitud'        : 'float64',
    'significancia'   : 'float64',
    'num_estaciones'  : 'float64',
    'brecha_azimutal' : 'float64',
    'tsunami'         : 'int8',      # Solo toma valores 0 o 1
})

# 2. Columnas de fecha para trazabilidad temporal y benchmarking
df_etl['fecha'] = pd.to_datetime(df_etl['tiempo_ms'], unit='ms')
df_etl['anio']  = df_etl['fecha'].dt.year

# 3a. Normalizar longitudes al rango estandar [-180, 180]
#     La API devuelve longitudes > 180 para cruzar el antimeridiano;
#     aqui se convierte de vuelta a la convencion estandar.
df_etl.loc[df_etl['longitud'] > 180, 'longitud'] -= 360

# 3b. Descartar registros con valores fisicamente imposibles
df_etl = df_etl[
    df_etl['magnitud'      ].between(5, 10)    &  # Rango valido de magnitud sismica
    df_etl['profundidad_km'].between(-5, 750)   &  # -5 permite flotabilidad superficial
    df_etl['latitud'       ].between(-90, 90)   &
    df_etl['longitud'      ].between(-180, 180)
]
# Clip a 0 km para profundidades negativas residuales (artefacto de localizacion)
df_etl['profundidad_km'] = df_etl['profundidad_km'].clip(lower=0)

print("DESPUES - tipos de datos normalizados:")
print(df_etl[['magnitud','profundidad_km','latitud','longitud',
              'significancia','num_estaciones','brecha_azimutal','tsunami']].dtypes.to_string())
print()
print("Rangos fisicos validados y longitudes normalizadas a [-180, 180]")


<br>

## 6.4 Exportacion del Dataset Limpio

<br>

In [ ]:
# =============================================================
#  SECCION 6.4 - EXPORTACION DEL DATASET LIMPIO
# =============================================================

RUTA_CLEAN = f"{DIRS['processed']}/usgs_clean.csv"
df_etl.to_csv(RUTA_CLEAN, index=False)

# Resumen comparativo ANTES vs DESPUES del pipeline ETL completo
print("=" * 58)
print("  RESUMEN ETL - ANTES Y DESPUES")
print("=" * 58)
print(f"  Filas originales    : {filas_inicial:>10,d}")
print(f"  Filas finales       : {len(df_etl):>10,d}")
print(f"  Filas descartadas   : {filas_inicial - len(df_etl):>10,d}"
      f"  ({(filas_inicial-len(df_etl))/filas_inicial*100:.1f}%)")
print(f"  Columnas            : {df_etl.shape[1]:>10,d}")
print(f"  Nulos en features   : {int(df_etl[['magnitud','profundidad_km','latitud','longitud','significancia','num_estaciones','brecha_azimutal','tsunami']].isna().sum().sum()):>10,d}")
print(f"  Exportado a         : {RUTA_CLEAN}")
print("=" * 58)

# Muestra de las primeras filas con nombres en espanol
print()
print("Muestra del dataset limpio (5 filas):")
display(df_etl[['magnitud','profundidad_km','latitud','longitud',
                'significancia','num_estaciones','brecha_azimutal',
                'tsunami','fecha']].head())


---

# SECCION 7 - ANALISIS EXPLORATORIO DE DATOS (EDA)

<br>

| Sub-seccion | Contenido |
|---|---|
| 7.1 | Estadisticas descriptivas |
| 7.2 | Distribucion de variables y del target |
| 7.3 | Porcentaje de nulos por variable candidata |
| 7.4 | Matriz de correlacion COMPLETA (todas las variables de la API) |
| 7.5 | Seleccion de features: por que estas 7 y por que NO las otras |
| 7.6 | Matriz de correlacion de las 7 features seleccionadas |

<br>

## 7.1 Estadisticas Descriptivas

<br>

In [ ]:
# =============================================================
#  SECCION 7.1 - ESTADISTICAS DESCRIPTIVAS
# =============================================================

TODAS_CANDIDATAS = [
    'magnitud', 'profundidad_km', 'latitud', 'longitud',
    'significancia', 'num_estaciones', 'brecha_azimutal',
    'dist_min_estacion', 'error_rms', 'reportes_sentido',
    'intensidad_cdi', 'intensidad_mmi'
]

print("Estadisticas descriptivas de todas las variables numericas candidatas:")
display(
    df_etl[TODAS_CANDIDATAS].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(2)
)

<br>

## 7.2 Distribucion de Variables y del Target

<br>

In [ ]:
# =============================================================
#  SECCION 7.2 - DISTRIBUCION DE VARIABLES Y TARGET
# =============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Distribucion de magnitud
axes[0].hist(df_etl['magnitud'].dropna(), bins=40,
             color='#2c7fb8', edgecolor='white')
axes[0].set_title('Distribucion de Magnitud (M >= 5.0)', fontweight='bold')
axes[0].set_xlabel('Magnitud')
axes[0].set_ylabel('Frecuencia')

# Distribucion de profundidad
axes[1].hist(df_etl['profundidad_km'].dropna(), bins=40,
             color='#41ab5d', edgecolor='white')
axes[1].set_title('Distribucion de Profundidad (km)', fontweight='bold')
axes[1].set_xlabel('Profundidad (km)')

# Distribucion del target (desbalance de clases)
conteo = df_etl['tsunami'].value_counts().sort_index()
axes[2].bar(['No tsunami (0)', 'Tsunami (1)'], conteo.values,
            color=['#74a9cf', '#d7301f'])
axes[2].set_title('Distribucion del Target: Clase Tsunami', fontweight='bold')
for i, v in enumerate(conteo.values):
    axes[2].text(i, v, f'{v:,}\n({v/len(df_etl)*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

ratio = conteo[0] / conteo[1]
print(f"Ratio de desbalance clase 0 / clase 1 = {ratio:.1f} : 1")
print("-> Justifica el uso de scale_pos_weight y AUC-ROC como metrica principal")

<br>

## 7.3 Porcentaje de Nulos por Variable Candidata

<br>

Esta grafica es clave para la decision de seleccion de features: variables con mas del 40% de nulos son practicamente inutilizables en un modelo sin introducir sesgo de imputacion.

<br>

In [ ]:
# =============================================================
#  SECCION 7.3 - PORCENTAJE DE NULOS POR VARIABLE CANDIDATA
# =============================================================

nulos_pct = (df_etl[TODAS_CANDIDATAS].isna().mean() * 100).sort_values(ascending=False).round(1)

colores = ['#d7301f' if v > 40 else '#fe9929' if v > 10 else '#2ca25f'
           for v in nulos_pct]

fig, ax = plt.subplots(figsize=(11, 4.5))
nulos_pct.plot(kind='bar', ax=ax, color=colores)
ax.axhline(40, color='#d7301f', ls='--', lw=1.5, label='Umbral de descarte (40%)')
ax.set_title('Porcentaje de Valores Nulos por Variable Candidata', fontweight='bold')
ax.set_ylabel('% Nulos')
ax.set_xlabel('')
plt.xticks(rotation=35, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

print("Detalle de nulos por variable:")
for var, pct in nulos_pct.items():
    barra = '#' * int(pct / 5)
    flag  = '  <- DESCARTAR' if pct > 40 else ''
    print(f"  {var:20s} : {pct:5.1f}%  {barra}{flag}")

<br>

## 7.4 Matriz de Correlacion COMPLETA

<br>

Esta es la matriz de correlacion de **todas las variables numericas candidatas** extraidas de la API, incluyendo las que posteriormente seran descartadas. Permite evaluar la relacion lineal de cada variable con el target `tsunami` y entre ellas, como insumo para la seleccion de features de la Seccion 7.5.

<br>

In [ ]:
# =============================================================
#  SECCION 7.4 - MATRIZ DE CORRELACION COMPLETA (todas las
#                variables numericas de la API + target)
# =============================================================

# Incluye las 12 variables candidatas + el target tsunami
VARS_COMPLETA = TODAS_CANDIDATAS + ['tsunami']

# Etiquetas legibles para el heatmap
ETIQUETAS_COMPLETA = {
    'magnitud'          : 'Magnitud',
    'profundidad_km'    : 'Profundidad',
    'latitud'           : 'Latitud',
    'longitud'          : 'Longitud',
    'significancia'     : 'Significancia',
    'num_estaciones'    : 'N. Estaciones',
    'brecha_azimutal'   : 'Brecha Azimutal',
    'dist_min_estacion' : 'Dist. Min. Estacion',
    'error_rms'         : 'Error RMS',
    'reportes_sentido'  : 'Reportes Sentido',
    'intensidad_cdi'    : 'Intensidad CDI',
    'intensidad_mmi'    : 'Intensidad MMI',
    'tsunami'           : 'TSUNAMI (target)',
}

# Conservar solo las que existen y son numericas
cols_validas  = [c for c in VARS_COMPLETA
                 if c in df_etl.columns
                 and pd.api.types.is_numeric_dtype(df_etl[c])]
df_corr_total = df_etl[cols_validas].rename(columns=ETIQUETAS_COMPLETA)
corr_total    = df_corr_total.corr(method='pearson')

plt.figure(figsize=(14, 11))
sns.heatmap(
    corr_total,
    annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Coeficiente de Pearson (r)'},
    annot_kws={'size': 9}
)
plt.title(
    'Matriz de Correlacion COMPLETA\nTodas las variables numericas de la API USGS + target tsunami',
    fontsize=13, fontweight='bold', pad=14
)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(f"{DIRS['processed']}/correlacion_completa.png", dpi=150, bbox_inches='tight')
plt.show()

print("Correlacion de TODAS las variables candidatas con el target tsunami (ordenada):")
target_col = 'TSUNAMI (target)'
serie_total = corr_total[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)
for v, r in serie_total.items():
    flag = '  <- LEAKAGE' if v in ['Reportes Sentido', 'Intensidad CDI', 'Intensidad MMI'] else ''
    print(f"   {v:<22} r = {r:+.3f}{flag}")

<br>

## 7.5 Seleccion de Features: Por que estas 7 y por que NO las otras

<br>

Con la matriz de correlacion completa, el analisis de nulos y el criterio anti-leakage, la decision por variable es:

**VARIABLES SELECCIONADAS (7 features del modelo):**

| Feature | Razon de inclusion |
|---|---|
| `magnitud` | Mayor correlacion individual con el target. Predictor fisico principal: la energia liberada determina el potencial de desplazamiento de la columna de agua. |
| `profundidad_km` | Correlacion negativa con el target, consistente con la fisica: solo sismos someros (< 70 km) transfieren energia al fondo oceanico. |
| `latitud` / `longitud` | Su correlacion lineal con el target es baja, pero su valor es NO LINEAL: XGBoost particiona el espacio geografico capturando zonas de subduccion vs fallas continentales. Cero nulos. |
| `significancia` | Alta correlacion con el target; integra magnitud + impacto reportado en un solo indicador robusto. |
| `num_estaciones` | Proxy de calidad de localizacion e instrumentacion. Nulos moderados imputables. |
| `brecha_azimutal` | Complementa a `num_estaciones` midiendo la geometria de la cobertura. |

**VARIABLES DESCARTADAS y su razon tecnica:**

| Variable | Razon del descarte |
|---|---|
| `reportes_sentido` (felt) | **LEAKAGE:** los reportes ciudadanos llegan horas despues del sismo. Un sistema de alerta no puede esperar esos datos. Ademas: >80% nulos. |
| `intensidad_cdi` | **LEAKAGE:** se calcula a partir de los reportes ciudadanos posteriores. >80% nulos. |
| `intensidad_mmi` | **LEAKAGE parcial:** la intensidad instrumental se publica tras el procesamiento. Correlaciona casi 1:1 con magnitud + profundidad (redundante). >85% nulos. |
| `nivel_alerta_pager` | **LEAKAGE directo y circular:** PAGER es el sistema de alerta del USGS emitido despues del evento. Usarlo seria predecir una alerta usando otra alerta. >90% nulos. |
| `dist_min_estacion` | Redundante con `num_estaciones` y `brecha_azimutal`; mayor porcentaje de nulos del trio y menor correlacion con el target. |
| `error_rms` | Metrica interna del localizador sin relacion fisica con la tsunamigenicidad. Correlacion nula con el target. |
| `tipo_magnitud` | Categorica casi constante (domina mww/mwb); sin poder discriminante. |
| `lugar`, `tiempo_ms`, `id` | Identificadores y texto descriptivo, no predictores. `tiempo_ms` solo se conserva para derivar `fecha` y `anio`. |

**Criterios aplicados en orden de prioridad:** (1) disponibilidad ANTES o en el instante del evento (sin leakage), (2) porcentaje de nulos menor al 40%, (3) relacion fisica o estadistica con el target, (4) no redundancia con otra feature ya incluida.

<br>

In [ ]:
# =============================================================
#  SECCION 7.5 - DEFINICION DE FEATURES Y TARGET DEL MODELO
# =============================================================
#  FEATURES (7 variables de entrada):
#    Seleccionadas por disponibilidad en el endpoint basico de
#    la API USGS, correlacion con el target y ausencia de leakage.
#    La justificacion completa esta en la Seccion 7.4.
#
#  TARGET (variable de salida):
#    La bandera oficial 'tsunami' del catalogo USGS:
#      0 -> el evento NO genero un tsunami documentado
#      1 -> el evento SI genero un tsunami documentado
# =============================================================

FEATURES = [
    'magnitud',         # Energia liberada (principal predictor)
    'profundidad_km',   # Profundidad del hipocentro (critica: sismos someros son mas tsunamigenicos)
    'latitud',          # Contexto tectonico geografico
    'longitud',         # Contexto tectonico geografico
    'significancia',    # Indice compuesto de impacto USGS
    'num_estaciones',   # Calidad de localizacion (cuantas estaciones lo midieron)
    'brecha_azimutal',  # Calidad de localizacion (cobertura angular de estaciones)
]

TARGET = 'tsunami'      # Variable binaria: 0 = no tsunami | 1 = tsunami

print(f"Features del modelo ({len(FEATURES)}):")
for f in FEATURES:
    print(f"  - {f}")
print()
print(f"Target: '{TARGET}'")
print()

# Variables excluidas y razon
excluidas = {
    'reportes_sentido' : 'Leakage - solo se reporta DESPUES de ocurrido el tsunami',
    'intensidad_cdi'   : 'Leakage - idem',
    'intensidad_mmi'   : 'Leakage - idem',
    'dist_min_estacion': 'Alta tasa de nulos (>40%) - ver Seccion 7.3',
    'error_rms'        : 'Correlacion baja con target - ver Seccion 7.4',
}
print("Variables excluidas:")
for var, razon in excluidas.items():
    print(f"  {var:<22} : {razon}")


<br>

## 7.6 Matriz de Correlacion de las 7 Features Seleccionadas

<br>

Esta segunda matriz muestra exclusivamente las relaciones entre las 7 features del modelo y el target, con mayor nivel de detalle y claridad visual.

<br>

In [ ]:
# =============================================================
#  SECCION 7.6 - MATRIZ DE CORRELACION DE LAS 7 FEATURES
#                SELECCIONADAS (para el modelo XGBoost)
# =============================================================

ETIQUETAS_MODELO = {
    'magnitud'       : 'Magnitud',
    'profundidad_km' : 'Profundidad',
    'latitud'        : 'Latitud',
    'longitud'       : 'Longitud',
    'significancia'  : 'Significancia',
    'num_estaciones' : 'N. Estaciones',
    'brecha_azimutal': 'Brecha Azimutal',
    'tsunami'        : 'Tsunami (target)',
}

features_modelo = FEATURES + [TARGET]
cols_m          = [c for c in features_modelo
                   if c in df_etl.columns
                   and pd.api.types.is_numeric_dtype(df_etl[c])]
df_corr_m       = df_etl[cols_m].rename(columns=ETIQUETAS_MODELO)
corr_modelo     = df_corr_m.corr(method='pearson')

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_modelo,
    annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Coeficiente de Pearson (r)'},
    annot_kws={'size': 10}
)
plt.title(
    'Matriz de Correlacion - Features Seleccionadas para el Modelo XGBoost',
    fontsize=12, fontweight='bold', pad=12
)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
plt.savefig(f"{DIRS['processed']}/correlacion_features_modelo.png", dpi=150, bbox_inches='tight')
plt.show()

print("Correlacion de los 7 features del modelo con el target tsunami:")
serie = corr_modelo['Tsunami (target)'].drop('Tsunami (target)').sort_values(key=lambda s: s.abs(), ascending=False)
for v, r in serie.items():
    print(f"   {v:<18} r = {r:+.3f}")

---

# SECCION 8 - BENCHMARKING DE ENGINES: PANDAS VS PYSPARK VS DASK

<br>

Se implementa el **mismo pipeline ETL** en los tres engines y se mide de forma estandarizada:

- **Tiempo de ejecucion** (`time.time()`)
- **Consumo de memoria RAM** (`psutil.Process`)

Operaciones del pipeline: (1) carga del CSV, (2) filtrado M >= 6.0, (3) agrupacion por anio, (4) join con los agregados.

<br>

## 8.1 Infraestructura de Medicion

<br>

In [ ]:
# =============================================================
#  SECCION 8.1 - INFRAESTRUCTURA DE MEDICION
# =============================================================

PROCESO          = psutil.Process()
RESULTADOS_BENCH = []


def medir(engine: str, operacion: str, funcion):
    '''
    Ejecuta una funcion midiendo tiempo de pared y delta de RAM.

    Parametros:
        engine    : str      - nombre del engine (Pandas, PySpark, Dask)
        operacion : str      - nombre de la operacion medida
        funcion   : callable - lambda que ejecuta la operacion

    Retorna el resultado de la funcion y registra las metricas en RESULTADOS_BENCH.
    '''
    ram_antes   = PROCESO.memory_info().rss / 1024**2
    t0          = time.time()
    resultado   = funcion()
    t1          = time.time()
    ram_despues = PROCESO.memory_info().rss / 1024**2

    RESULTADOS_BENCH.append({
        'Engine'         : engine,
        'Operacion'      : operacion,
        'Tiempo (s)'     : round(t1 - t0, 4),
        'RAM delta (MB)' : round(ram_despues - ram_antes, 2),
    })

    print(f"  [{engine:<7}] {operacion:<12} -> {t1-t0:7.4f}s  |  RAM {ram_despues-ram_antes:+8.2f} MB")
    return resultado

<br>

## 8.2 Pipeline Pandas

<br>

In [ ]:
# =============================================================
#  SECCION 8.2 - PIPELINE ETL EN PANDAS
# =============================================================
#  Medicion de: carga, filtrado, agrupacion y join
# =============================================================

print("=== BENCHMARK PANDAS ===")

pdf         = medir('Pandas', 'carga',
                    lambda: pd.read_csv(RUTA_CLEAN))
pdf_filtrado = medir('Pandas', 'filtrado',
                     lambda: pdf[pdf['magnitud'] >= 6.0])
pdf_agrupado = medir('Pandas', 'agrupacion',
                     lambda: pdf_filtrado.groupby('anio')
                                         .agg(mag_promedio=('magnitud', 'mean'),
                                              eventos=('id', 'count'))
                                         .reset_index())
pdf_join     = medir('Pandas', 'join',
                     lambda: pdf_filtrado.merge(pdf_agrupado, on='anio', how='left'))

print()
print("Resultado (agrupacion por anio - primeras 5 filas):")
display(pdf_agrupado.head())

<br>

## 8.3 Pipeline PySpark

<br>

**Diferencias de sintaxis PySpark vs Pandas (SEIS-68):**

| Operacion | Pandas | PySpark |
|---|---|---|
| Filtrado | `df[df['col'] >= val]` | `df.filter(F.col('col') >= val)` |
| Agrupacion | `df.groupby().agg(...)` | `df.groupBy().agg(F.mean(...).alias(...))` |
| Join | `df.merge(otro, on='col')` | `df.join(otro, on='col', how='left')` |
| Ejecucion | Eager (inmediata) | Lazy - se materializa con acciones (`.count()`, `.show()`) |

<br>

In [ ]:
# =============================================================
#  SECCION 8.3 - PIPELINE ETL EN PYSPARK
# =============================================================
#  NOTA: Spark es lazy. Se fuerza la materializacion con .count()
#        para que el tiempo medido sea real, no solo el plan.
# =============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName('SeismicPipeline-Benchmark')
         .config('spark.driver.memory', '4g')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print(f"SparkSession iniciada - version: {spark.version}")
print()
print("=== BENCHMARK PYSPARK ===")

sdf = medir('PySpark', 'carga',
            lambda: spark.read.csv(RUTA_CLEAN, header=True, inferSchema=True))
sdf.cache()
sdf.count()                              # Materializa el cache

sdf_filtrado = medir('PySpark', 'filtrado',
                     lambda: (lambda d: (d.count(), d)[1])
                             (sdf.filter(F.col('magnitud') >= 6.0)))
sdf_agrupado = medir('PySpark', 'agrupacion',
                     lambda: (lambda d: (d.count(), d)[1])(
                         sdf_filtrado
                         .groupBy('anio')
                         .agg(F.mean('magnitud').alias('mag_promedio'),
                              F.count('id').alias('eventos'))))
sdf_join     = medir('PySpark', 'join',
                     lambda: (lambda d: (d.count(), d)[1])
                             (sdf_filtrado.join(sdf_agrupado, on='anio', how='left')))

<br>

## 8.4 Pipeline Dask

<br>

**Comportamiento lazy de Dask:** las operaciones construyen un grafo de tareas en memoria; el calculo real ocurre solo al llamar a `.compute()`. Esto contrasta con Pandas (eager) y permite potencialmente escalar a multiples nucleos o nodos.

**Nota tecnica sobre la inferencia de tipos en Dask:** Dask infiere los tipos leyendo solo una muestra del inicio del CSV. Columnas como `nivel_alerta_pager` (texto que empieza vacia) son incorrectamente inferidas como `float64`. Se especifican los tipos manualmente para evitar el error, lo que es una diferencia practica importante frente a Pandas que lee el archivo completo.

<br>

In [ ]:
# =============================================================
#  SECCION 8.4 - PIPELINE ETL EN DASK
# =============================================================
#  NOTA TECNICA: Dask infiere dtypes por muestreo (lectura lazy).
#  Columnas de texto con muchos nulos al inicio del CSV son mal
#  inferidas como float64 y fallan en .compute(). Se especifican
#  manualmente (diferencia practica documentada vs Pandas).
# =============================================================

import dask.dataframe as dd

# Tipos que Dask no puede inferir correctamente por muestreo
DTYPES_DASK = {
    'nivel_alerta_pager' : 'object',
    'tipo_magnitud'      : 'object',
    'lugar'              : 'object',
    'id'                 : 'object',
}

print("=== BENCHMARK DASK ===")

ddf = medir('Dask', 'carga',
            lambda: dd.read_csv(RUTA_CLEAN, dtype=DTYPES_DASK))

# Demostrar el comportamiento lazy
grafo = ddf[ddf['magnitud'] >= 6.0]
print()
print(f"  Objeto lazy (sin computar): {type(grafo).__name__}")
print("  -> Las operaciones construyen un grafo de tareas en memoria.")
print("     El calculo ocurre solo al llamar a .compute()")
print()

ddf_filtrado = medir('Dask', 'filtrado',
                     lambda: ddf[ddf['magnitud'] >= 6.0].compute())
ddf_agrupado = medir('Dask', 'agrupacion',
                     lambda: (dd.from_pandas(ddf_filtrado, npartitions=4)
                              .groupby('anio')
                              .agg({'magnitud': 'mean', 'id': 'count'})
                              .compute()
                              .rename(columns={'magnitud': 'mag_promedio', 'id': 'eventos'})
                              .reset_index()))
ddf_join     = medir('Dask', 'join',
                     lambda: (dd.from_pandas(ddf_filtrado, npartitions=4)
                              .merge(dd.from_pandas(ddf_agrupado, npartitions=1),
                                     on='anio', how='left')
                              .compute()))

<br>

## 8.5 Resultados Comparativos

<br>

In [ ]:
# =============================================================
#  SECCION 8.5 - TABLA COMPARATIVA Y GRAFICA
# =============================================================

tabla_bench = pd.DataFrame(RESULTADOS_BENCH)

# Tabla de tiempos pivoteada
tabla_pivot = (tabla_bench
               .pivot_table(index='Operacion', columns='Engine', values='Tiempo (s)')
               .reindex(['carga', 'filtrado', 'agrupacion', 'join']))
tabla_pivot.loc['TOTAL'] = tabla_pivot.sum()

print("=== TIEMPOS DE EJECUCION (segundos) ===")
display(tabla_pivot.round(4))

# Tabla de memoria
tabla_ram = (tabla_bench
             .pivot_table(index='Operacion', columns='Engine', values='RAM delta (MB)')
             .reindex(['carga', 'filtrado', 'agrupacion', 'join']))

print()
print("=== CONSUMO DE MEMORIA - DELTA (MB) ===")
display(tabla_ram.round(2))

# Grafica comparativa
fig, ax = plt.subplots(figsize=(11, 5))
tabla_pivot.drop('TOTAL').plot(
    kind='bar', ax=ax,
    color={'Pandas': '#2c7fb8', 'PySpark': '#e6550d', 'Dask': '#31a354'}
)
ax.set_title('Benchmarking de Tiempos por Operacion: Pandas vs PySpark vs Dask',
             fontweight='bold')
ax.set_ylabel('Tiempo (s)')
ax.set_xlabel('Operacion del Pipeline ETL')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<br>

## 8.6 Conclusion del Benchmarking

<br>

Para el volumen de datos de este proyecto (decenas de miles de eventos M >= 5.0, < 100 MB en disco), **Pandas es el engine recomendado** por tres razones:

1. **Overhead vs beneficio:** PySpark inicializa la JVM y Dask construye un grafo de tareas distribuido; ambos costos fijos son significativos para datasets que caben en RAM de una sola maquina.
2. **Inferencia de tipos:** Pandas lee el archivo completo y detecta correctamente todos los tipos. Dask, al muestrear, fallo con la columna `nivel_alerta_pager` - diferencia practica documentada en la seccion anterior.
3. **Escalabilidad futura:** si el proyecto escalara a catalogos sin filtro de magnitud (millones de eventos) o a datos de waveforms, **Dask** seria la transicion natural por mantener la API de Pandas. **PySpark** seria la opcion para un cluster real con decenas de GB.

<br>

---

# SECCION 9 - PREPARACION DEL DATASET DE MACHINE LEARNING

<br>

## 9.1 Construccion del Dataset Final

<br>

In [ ]:
# =============================================================
#  SECCION 9.1 - DATASET FINAL DE ML
# =============================================================

df_ml = df_etl[FEATURES + [TARGET, 'fecha', 'lugar', 'id']].copy()

# Verificacion de integridad del target
assert df_ml[TARGET].isna().sum()   == 0,    "Target con nulos"
assert set(df_ml[TARGET].unique()) <= {0, 1}, "Target con valores invalidos"

RUTA_ML = f"{DIRS['processed']}/dataset_ml.csv"
df_ml.to_csv(RUTA_ML, index=False)

dist = df_ml[TARGET].value_counts().sort_index()

print("=" * 58)
print("  DATASET FINAL DE MACHINE LEARNING")
print("=" * 58)
print(f"  Forma           : {df_ml.shape[0]:,} filas x {df_ml.shape[1]} columnas")
print(f"  Features        : {len(FEATURES)}")
print(f"  Target verificado: sin nulos, valores en {{0, 1}}")
print()
print(f"  Distribucion del target:")
print(f"    tsunami = 0 : {dist[0]:,} ({dist[0]/len(df_ml)*100:.2f}%)")
print(f"    tsunami = 1 : {dist[1]:,} ({dist[1]/len(df_ml)*100:.2f}%)")
print(f"  Exportado a     : {RUTA_ML}")
print("=" * 58)

<br>

## 9.2 Split Estratificado 80/20 y Calculo de scale_pos_weight

<br>

In [ ]:
# =============================================================
#  SECCION 9.2 - SPLIT ESTRATIFICADO 80/20 Y scale_pos_weight
# =============================================================
#  Split estratificado: garantiza que la proporcion de clase 1
#  (eventos tsunamigenicos) sea identica en train y test.
#  Esto es critico con datos desbalanceados: sin estratificacion,
#  el conjunto de test podria tener 0 tsunamis por azar.
#
#  scale_pos_weight: le dice a XGBoost cuanto mas pesar los
#  errores en la clase minoritaria (tsunami=1) durante el
#  entrenamiento. Valor = n_clase_0 / n_clase_1.
# =============================================================

X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,         # Preserva la proporcion de clases en train y test
    random_state = SEED,
)

# Calculo del peso corrector para el desbalance de clases
n0               = int((y_train == 0).sum())   # Eventos sin tsunami en train
n1               = int((y_train == 1).sum())   # Eventos con tsunami en train
SCALE_POS_WEIGHT = n0 / n1                     # Factor de correccion para XGBoost

print("=" * 58)
print("  SPLIT ESTRATIFICADO 80/20")
print("=" * 58)
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print()
print(f"  Clase 0 (no tsunami) en train : {n0:,}")
print(f"  Clase 1 (tsunami)    en train : {n1:,}")
print(f"  scale_pos_weight = {n0:,} / {n1:,} = {SCALE_POS_WEIGHT:.2f}")
print()
print("  Verificacion de estratificacion (proporciones deben coincidir):")
verif = pd.DataFrame({
    'Train': y_train.value_counts(normalize=True).sort_index() * 100,
    'Test' : y_test.value_counts(normalize=True).sort_index()  * 100,
}).round(2)
verif.index = ['Clase 0 (%)', 'Clase 1 (%)']
print(verif.to_string())
print("=" * 58)


---

# SECCION 10 - ENTRENAMIENTO DEL CLASIFICADOR XGBOOST

<br>

## 10.1 Configuracion e Instanciacion del Modelo

<br>

In [ ]:
# =============================================================
#  SECCION 10.1 - INSTANCIACION DEL XGBOOST
# =============================================================
#  objective='binary:logistic' : clasificacion binaria
#  scale_pos_weight            : correccion del desbalance en la perdida
#  eval_metric='auc'           : metrica principal monitoreada en eval_set
# =============================================================

modelo_xgb = XGBClassifier(
    objective       = 'binary:logistic',
    scale_pos_weight= SCALE_POS_WEIGHT,    # Correccion de desbalance de clases
    n_estimators    = 300,                 # Numero de arboles del ensamble
    max_depth       = 6,                   # Profundidad maxima de cada arbol
    learning_rate   = 0.1,                 # Tasa de aprendizaje (eta)
    subsample       = 0.9,                 # Fraccion de filas por arbol
    colsample_bytree= 0.9,                 # Fraccion de features por arbol
    eval_metric     = 'auc',
    random_state    = SEED,
    n_jobs          = -1,
)

print("Modelo XGBoost instanciado:")
print(modelo_xgb)

<br>

## 10.2 Entrenamiento con Monitoreo por Iteracion

<br>

In [ ]:
# =============================================================
#  SECCION 10.2 - ENTRENAMIENTO CON eval_set
# =============================================================
#  eval_set permite monitorear el AUC en train y test por cada
#  iteracion. Si la curva de test baja mientras la de train sube,
#  hay overfitting (ver Seccion 11 para el diagnostico completo).
# =============================================================

modelo_xgb.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_test, y_test)],
    verbose  = 50,     # Imprime el AUC cada 50 iteraciones
)

print()
print("Entrenamiento completado")

In [ ]:
# =============================================================
#  SECCION 10.3 - VERIFICACION SOBRE 5 REGISTROS DEL TEST
# =============================================================

muestra_idx = X_test.sample(5, random_state=SEED).index
muestra     = df_ml.loc[muestra_idx, ['lugar', 'magnitud', 'profundidad_km']].copy()
muestra['tsunami_real']   = y_test.loc[muestra_idx].values
muestra['prob_tsunami']   = modelo_xgb.predict_proba(X_test.loc[muestra_idx])[:, 1].round(4)
muestra['prediccion_bin'] = modelo_xgb.predict(X_test.loc[muestra_idx])

print("Verificacion sobre 5 registros del conjunto de test:")
display(muestra)

In [ ]:
# =============================================================
#  SECCION 10.4 - SERIALIZACION DEL MODELO
# =============================================================

RUTA_MODELO = f"{DIRS['models']}/xgboost_model.pkl"
joblib.dump(modelo_xgb, RUTA_MODELO)

print(f"Modelo serializado en : {RUTA_MODELO}")
print(f"Tamanio               : {os.path.getsize(RUTA_MODELO)/1024**2:.2f} MB")
print()
print("Para cargar el modelo en otro notebook:")
print("  import joblib")
print(f"  modelo = joblib.load('{RUTA_MODELO}')")

---

# SECCION 11 - DIAGNOSTICO DE ENTRENAMIENTO

<br>

Un modelo bien entrenado cumple tres condiciones verificables simultaneamente:

| Condicion | Como se verifica | Que buscamos |
|---|---|---|
| **No underfitting** | AUC en train | Valor alto (> 0.85) |
| **No overfitting** | Brecha AUC train - test | Menor a 0.05 (ideal), aceptable hasta 0.10 |
| **Estabilidad** | Validacion cruzada 5-fold | Desviacion estandar < 0.02 |

<br>

## 11.1 Curva de Aprendizaje (eval_set)

<br>

In [ ]:
# =============================================================
#  SECCION 11.1 - CURVA DE APRENDIZAJE TRAIN VS TEST
# =============================================================
#  Lectura diagnostica:
#  - Ambas curvas BAJAS    -> underfitting
#  - Test baja mientras    -> overfitting
#    train sube (brecha creciente)
#  - Ambas altas, brecha   -> bien entrenado
#    pequenia y estable
# =============================================================

historial  = modelo_xgb.evals_result()
auc_train  = historial['validation_0']['auc']
auc_test   = historial['validation_1']['auc']

plt.figure(figsize=(10, 5))
plt.plot(auc_train, label='AUC - Entrenamiento (train)', color='#2c7fb8', lw=2)
plt.plot(auc_test,  label='AUC - Evaluacion (test)',     color='#d7301f', lw=2)
plt.fill_between(range(len(auc_train)), auc_test, auc_train,
                 alpha=0.12, color='gray',
                 label='Brecha train-test (overfitting si crece)')
plt.xlabel('Iteracion (arbol agregado al ensamble)')
plt.ylabel('AUC-ROC')
plt.title('Curva de aprendizaje XGBoost\nDiagnostico de underfitting / overfitting',
          fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

<br>

## 11.2 Brecha Numerica y Diagnostico Automatico

<br>

In [ ]:
# =============================================================
#  SECCION 11.2 - BRECHA TRAIN-TEST Y DIAGNOSTICO
# =============================================================

auc_en_train = roc_auc_score(y_train, modelo_xgb.predict_proba(X_train)[:, 1])
auc_en_test  = roc_auc_score(y_test,  modelo_xgb.predict_proba(X_test)[:, 1])
brecha       = auc_en_train - auc_en_test

print("=" * 55)
print("  DIAGNOSTICO DE ENTRENAMIENTO - Brecha Train/Test")
print("=" * 55)
print(f"  AUC-ROC en train  : {auc_en_train:.4f}")
print(f"  AUC-ROC en test   : {auc_en_test:.4f}")
print(f"  Brecha            : {brecha:.4f}")
print()
if brecha < 0.05:
    print("  DIAGNOSTICO: BIEN ENTRENADO")
    print("  Brecha < 0.05 -> generalizacion adecuada, sin sobreajuste.")
elif brecha < 0.10:
    print("  DIAGNOSTICO: SOBREAJUSTE LEVE")
    print("  Brecha 0.05-0.10 -> aceptable pero monitorear.")
else:
    print("  DIAGNOSTICO: SOBREAJUSTE SIGNIFICATIVO")
    print("  Brecha > 0.10 -> reducir max_depth o subir regularizacion.")
print("=" * 55)

<br>

## 11.3 Validacion Cruzada Estratificada (5 Folds)

<br>

In [ ]:
# =============================================================
#  SECCION 11.3 - VALIDACION CRUZADA 5-FOLD
# =============================================================
#  Una desviacion estandar baja (< 0.02) entre folds demuestra
#  que el rendimiento es estable e independiente del split.
# =============================================================

cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modelo_cv = XGBClassifier(
    objective       = 'binary:logistic',
    scale_pos_weight= SCALE_POS_WEIGHT,
    n_estimators    = 300, max_depth=6, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9, eval_metric='auc',
    random_state    = SEED, n_jobs=-1,
)
scores_cv = cross_val_score(modelo_cv, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print("=" * 55)
print("  VALIDACION CRUZADA ESTRATIFICADA - 5 Folds")
print("=" * 55)
for i, s in enumerate(scores_cv, 1):
    print(f"  Fold {i}: AUC = {s:.4f}")
print()
print(f"  Media      : {scores_cv.mean():.4f}")
print(f"  Desv. std  : {scores_cv.std():.4f}")
print()
if scores_cv.std() < 0.02:
    print("  DIAGNOSTICO: MODELO ESTABLE")
    print("  Desv. std < 0.02 -> rendimiento consistente en todos los folds.")
else:
    print("  DIAGNOSTICO: VARIABILIDAD MODERADA")
    print("  Desv. std >= 0.02 -> rendimiento variable segun el split.")
print("=" * 55)

---

# SECCION 12 - EVALUACION DEL MODELO

<br>

## 12.1 AUC-ROC y Curva ROC

<br>

In [ ]:
# =============================================================
#  SECCION 12.1 - AUC-ROC Y CURVA ROC
# =============================================================

y_proba  = modelo_xgb.predict_proba(X_test)[:, 1]
y_pred   = modelo_xgb.predict(X_test)
AUC_XGB  = roc_auc_score(y_test, y_proba)
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#d7301f', lw=2,
         label=f'XGBoost (AUC = {AUC_XGB:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1,
         label='Clasificador aleatorio (AUC = 0.50)')
plt.fill_between(fpr, tpr, alpha=0.10, color='#d7301f')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (Recall / TPR)')
plt.title('Curva ROC - Clasificador XGBoost de Riesgo de Tsunami', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"AUC-ROC sobre el conjunto de test = {AUC_XGB:.4f}")

<br>

**Justificacion de AUC-ROC como metrica principal:**

Con un desbalance severo del target (clase 1 es una fraccion minima del dataset), el **accuracy es enganioso**: un modelo trivial que prediga siempre "no tsunami" alcanzaria mas del 90% de accuracy siendo completamente inutil. El AUC-ROC evalua la capacidad del modelo de **ordenar** correctamente los eventos por riesgo y es **independiente del umbral de decision** y **robusta al desbalance**, que es exactamente lo que necesita un sistema de score continuo de riesgo por provincia.

<br>

## 12.2 Matriz de Confusion

<br>

In [ ]:
# =============================================================
#  SECCION 12.2 - MATRIZ DE CONFUSION
# =============================================================
#  En este problema, los errores NO tienen el mismo costo:
#
#  FN (Falso Negativo): predijo "no tsunami" y hubo tsunami
#     -> El mas peligroso: no se emite alerta, vidas en riesgo.
#
#  FP (Falso Positivo): predijo "tsunami" y no lo hubo
#     -> Costo social (evacuaciones innecesarias), pero tolerable.
#
#  Un buen modelo para alertas debe minimizar FN sobre FP.
# =============================================================

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No tsunami (0)', 'Tsunami (1)'])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
disp.plot(ax=ax, cmap='Blues', values_format=',d')
ax.set_title('Matriz de confusion - Clasificador XGBoost', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("=" * 58)
print("  INTERPRETACION DE LA MATRIZ DE CONFUSION")
print("=" * 58)
print(f"  TN = {tn:,}  Sismos sin tsunami descartados correctamente")
print(f"  FP = {fp:,}   Falsas alarmas (tolerable - costo social)")
print(f"  FN = {fn:,}   CRITICO: tsunamis NO detectados (riesgo de vidas)")
print(f"  TP = {tp:,}   Tsunamis correctamente alertados")
print()
if fn == 0:
    print("  -> FN = 0: el modelo no omitio ningun tsunami en el test set.")
else:
    pct_fn = fn / (fn + tp) * 100
    print(f"  -> Tasa de FN = {pct_fn:.1f}% de los tsunamis reales no fueron detectados.")
print("=" * 58)


<br>

## 12.3 Metricas Completas

<br>

In [ ]:
# =============================================================
#  SECCION 12.3 - METRICAS COMPLETAS DEL MODELO
# =============================================================

print("Reporte completo de clasificacion:")
print()
print(classification_report(y_test, y_pred,
                             target_names=['No tsunami (0)', 'Tsunami (1)'],
                             digits=4))

METRICAS_XGB = {
    'AUC-ROC'   : round(AUC_XGB, 4),
    'Precision' : round(precision_score(y_test, y_pred), 4),
    'Recall'    : round(recall_score(y_test, y_pred), 4),
    'F1-score'  : round(f1_score(y_test, y_pred), 4),
}
print("Resumen:")
for k, v in METRICAS_XGB.items():
    print(f"  {k:<12} : {v}")

---

# SECCION 13 - MODELOS BASELINE DE COMPARACION

<br>

> **Nota de alcance:** Random Forest y Regresion Logistica se entrenan unicamente como **puntos de comparacion** para justificar la seleccion de XGBoost. Se usan exactamente los mismos datos, features y split (mismo `SEED`), garantizando una comparacion justa. El modelo del proyecto sigue siendo el XGBoost de la Seccion 10.

<br>

## 13.1 Entrenamiento de los Baselines

<br>

In [ ]:
# =============================================================
#  SECCION 13.1 - ENTRENAMIENTO DE BASELINES
# =============================================================

# --- Random Forest (class_weight='balanced' equivale conceptualmente a scale_pos_weight) ---
modelo_rf = RandomForestClassifier(
    n_estimators = 300,
    max_depth    = None,
    class_weight = 'balanced',
    random_state = SEED,
    n_jobs       = -1,
)
modelo_rf.fit(X_train, y_train)
proba_rf = modelo_rf.predict_proba(X_test)[:, 1]
pred_rf  = modelo_rf.predict(X_test)
print(f"Random Forest entrenado     | AUC-ROC = {roc_auc_score(y_test, proba_rf):.4f}")

# --- Regresion Logistica (requiere escalado de features) ---
modelo_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(class_weight='balanced',
                                  max_iter=2000, random_state=SEED)),
])
modelo_lr.fit(X_train, y_train)
proba_lr = modelo_lr.predict_proba(X_test)[:, 1]
pred_lr  = modelo_lr.predict(X_test)
print(f"Regresion Logistica entrenada | AUC-ROC = {roc_auc_score(y_test, proba_lr):.4f}")

<br>

## 13.2 Tabla Comparativa y Curvas ROC Superpuestas

<br>

In [ ]:
# =============================================================
#  SECCION 13.2 - TABLA COMPARATIVA Y CURVAS ROC
# =============================================================

def fila_metricas(nombre, y_true, y_p, proba):
    return {
        'Modelo'    : nombre,
        'AUC-ROC'   : round(roc_auc_score(y_true, proba), 4),
        'Precision' : round(precision_score(y_true, y_p), 4),
        'Recall'    : round(recall_score(y_true, y_p),    4),
        'F1-score'  : round(f1_score(y_true, y_p),        4),
    }

tabla_modelos = pd.DataFrame([
    fila_metricas('XGBoost (modelo del proyecto)', y_test, y_pred,  y_proba),
    fila_metricas('Random Forest (baseline)',      y_test, pred_rf, proba_rf),
    fila_metricas('Reg. Logistica (baseline)',     y_test, pred_lr, proba_lr),
]).set_index('Modelo')

print("Tabla comparativa de modelos:")
display(tabla_modelos)

# Curvas ROC superpuestas
plt.figure(figsize=(8, 6))
for nombre, proba, color in [
    ('XGBoost',        y_proba,  '#d7301f'),
    ('Random Forest',  proba_rf, '#2c7fb8'),
    ('Reg. Logistica', proba_lr, '#31a354'),
]:
    f, t, _ = roc_curve(y_test, proba)
    plt.plot(f, t, lw=2, color=color,
             label=f'{nombre} (AUC = {roc_auc_score(y_test, proba):.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Comparacion de Curvas ROC: XGBoost vs Baselines', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

<br>

## 13.3 Analisis Critico: Por que XGBoost supera a los baselines

<br>

1. **Frente a Regresion Logistica:** el riesgo de tsunami no es lineal. La interaccion magnitud-profundidad (M 7.5 somero es tsunamigenico, M 7.5 a 500 km no) y la dependencia espacial (latitud/longitud codifican zonas de subduccion) son relaciones que un modelo lineal no puede representar. XGBoost las captura mediante particiones jerarquicas sucesivas del espacio de features.

2. **Frente a Random Forest:** RF entrena arboles independientes en paralelo (bagging); XGBoost los entrena secuencialmente sobre los residuos del ensamble anterior (boosting con gradiente), concentrando capacidad de modelo en los casos dificiles, que en este problema son los escasos positivos. Ademas XGBoost ofrece `scale_pos_weight` integrado en la funcion de perdida, regularizacion L1/L2 explicita y monitoreo nativo con `eval_set`.

3. **Evidencia empirica:** la tabla y las curvas ROC confirman que XGBoost obtiene el mayor AUC-ROC bajo condiciones identicas de comparacion, validando la decision con datos del propio proyecto.

<br>

---

# SECCION 14 - JUSTIFICACION TECNICA DE LA SELECCION DE XGBOOST

<br>

## 14.1 Tabla de Ventajas y Desventajas

<br>

| Criterio | XGBoost | Random Forest | Reg. Logistica |
|---|---|---|---|
| Relaciones no lineales e interacciones | Excelente | Buena | No las captura |
| Manejo de desbalance | `scale_pos_weight` nativo en la perdida | Solo `class_weight` / resampling | Solo `class_weight` / resampling |
| Regularizacion | L1 + L2 + gamma + learning rate | Limitada | L1/L2 |
| Riesgo de overfitting | Controlable con eval_set | Bajo-medio | Bajo (underfitting aqui) |
| Velocidad | Alta (paralelizado, histogramas) | Media | Muy alta |
| Interpretabilidad | Feature importance + SHAP | Feature importance | Coeficientes directos |
| Rendimiento en tabulares desbalanceados | **Estado del arte** | Competitivo | Baseline |

<br>

## 14.2 Feature Importance

<br>

In [ ]:
# =============================================================
#  SECCION 14.2 - FEATURE IMPORTANCE
# =============================================================

importancias = pd.Series(
    modelo_xgb.feature_importances_, index=FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
xgb.plot_importance(
    modelo_xgb, ax=ax, importance_type='gain',
    title='Feature Importance (Gain) - Modelo XGBoost',
    xlabel='Ganancia promedio al usar la variable en splits'
)
plt.tight_layout()
plt.show()

print("Importancia relativa de cada feature:")
for feat, imp in importancias.items():
    barra = '#' * int(imp * 100 / importancias.max())
    print(f"  {feat:<18} : {imp:.4f}  {barra}")

<br>

## 14.3 Justificacion Tecnica (300+ palabras)

<br>

XGBoost (eXtreme Gradient Boosting) fue seleccionado como algoritmo del proyecto por la convergencia de cuatro razones tecnicas verificables en este dominio.

**Primera razon: naturaleza no lineal del problema.** La generacion de un tsunami depende de interacciones no lineales entre variables fisicas. La literatura sismologica establece que la tsunamigenicidad requiere simultaneamente magnitud alta, hipocentro somero y localizacion en zonas de subduccion oceanica; ninguna de estas condiciones es suficiente por separado. Los ensambles de arboles con boosting de gradiente representan estas conjunciones mediante particiones jerarquicas del espacio de features, mientras que un modelo lineal las pierde por construccion (Chen y Guestrin, 2016).

**Segunda razon: desbalance extremo del target.** XGBoost incorpora el hiperparametro `scale_pos_weight` directamente en la funcion de perdida logistica, reponderando el gradiente de los ejemplos positivos durante la optimizacion. Esto evita recurrir a tecnicas de remuestreo sintetico (SMOTE) que distorsionan la distribucion espacial real de los epicentros, un riesgo documentado cuando las features tienen significado geografico.

**Tercera razon: control del sobreajuste.** La regularizacion L1/L2 sobre los pesos de las hojas, el parametro gamma de complejidad minima por split, el submuestreo de filas y columnas, y el monitoreo por iteracion via `eval_set` permiten verificar empiricamente (Seccion 11) que la brecha train-test del AUC se mantiene acotada.

**Cuarta razon: evidencia empirica del propio proyecto.** En la comparacion controlada de la Seccion 13, con identicos datos, features, split y semilla aleatoria, XGBoost obtuvo el mayor AUC-ROC frente a Random Forest y Regresion Logistica, confirmando que la eleccion no es solo teorica sino medida con datos propios.

En conjunto, XGBoost ofrece el mejor equilibrio entre capacidad de representacion, manejo nativo del desbalance, control del sobreajuste y costo computacional para un dataset tabular de este volumen ejecutado en Google Colab.

**Referencias:** Chen, T., y Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD*, 785-794. https://doi.org/10.1145/2939672.2939785. Satake, K. (2014). Advances in earthquake and tsunami sciences. *Geoscience Letters, 1*(15). https://doi.org/10.1186/s40562-014-0015-7

<br>

---

# SECCION 15 - PROYECCION DE RIESGO SOBRE ECUADOR

<br>

El modelo se aplica al registro sismico costero ecuatoriano del propio dataset. Para cada sismo se calcula `predict_proba` y se proyecta a las 6 provincias costeras mediante un score de impacto que combina la probabilidad del modelo con la distancia Haversine epicentro-provincia.

<br>

## 15.1 Centroides de las Provincias Costeras

<br>

In [ ]:
# =============================================================
#  SECCION 15.1 - CENTROIDES DE LAS 6 PROVINCIAS COSTERAS
# =============================================================

PROVINCIAS = {
    'Esmeraldas' : ( 0.97, -79.65),
    'Manabi'     : (-0.95, -80.73),
    'Santa Elena': (-2.23, -80.86),
    'Guayas'     : (-2.62, -80.39),
    'El Oro'     : (-3.27, -80.05),
    'Galapagos'  : (-0.74, -90.31),
}

df_prov = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in PROVINCIAS.items()],
    columns=['Provincia', 'Latitud', 'Longitud']
)
display(df_prov)

<br>

## 15.2 Funciones de Proyeccion y Sistema de Alertas

<br>

In [ ]:
# =============================================================
#  SECCION 15.2 - FUNCIONES DE PROYECCION
# =============================================================

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    '''
    Calcula la distancia ortodromica entre dos puntos geograficos.

    Parametros:
        lat1, lon1 : float - coordenadas del punto 1 (epicentro)
        lat2, lon2 : float - coordenadas del punto 2 (provincia)

    Retorna:
        Distancia en kilometros (float).
    '''
    R    = 6371.0
    p1   = math.radians(lat1)
    p2   = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a    = math.sin(dphi/2)**2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def clasificar_alerta(score: float) -> str:
    '''Asigna nivel de alerta segun el score de impacto.'''
    if score >= 0.60: return 'ROJO'
    if score >= 0.40: return 'NARANJA'
    if score >= 0.20: return 'AMARILLO'
    return 'VERDE'


def proyectar_alertas(evento) -> pd.DataFrame:
    '''
    Proyecta las probabilidades de tsunami de un sismo a las 6 provincias.

    El score de impacto combina la probabilidad del modelo con un factor de
    atenuacion exponencial por distancia (decaimiento a 450 km).

    Retorna:
        pd.DataFrame ordenado por score_impacto descendente.
    '''
    filas = []
    for prov, (plat, plon) in PROVINCIAS.items():
        d         = haversine_km(evento['latitud'], evento['longitud'], plat, plon)
        atenuacion= math.exp(-d / 450)
        score     = evento['prob_tsunami'] * atenuacion
        filas.append({
            'Provincia'     : prov,
            'Distancia (km)': round(d, 1),
            'Prob. modelo'  : round(evento['prob_tsunami'], 4),
            'Score impacto' : round(score, 4),
            'Alerta'        : clasificar_alerta(score),
        })
    return pd.DataFrame(filas).sort_values('Score impacto', ascending=False).reset_index(drop=True)


print("Niveles de alerta configurados:")
print("  VERDE    : score < 0.20")
print("  AMARILLO : 0.20 <= score < 0.40")
print("  NARANJA  : 0.40 <= score < 0.60")
print("  ROJO     : score >= 0.60")

<br>

## 15.3 Validacion con Pedernales 2016 (Mw 7.8)

<br>

In [ ]:
# =============================================================
#  SECCION 15.3 - VALIDACION CON EL TERREMOTO DE PEDERNALES 2016
# =============================================================

# Filtrar el registro sismico ecuatoriano del dataset
mask_ec   = (df_ml['latitud'].between(-5.5, 2.5)) & (df_ml['longitud'].between(-93, -78))
df_ecuador= df_ml[mask_ec].copy()
df_ecuador['prob_tsunami'] = modelo_xgb.predict_proba(df_ecuador[FEATURES])[:, 1]

print(f"Sismos del registro ecuatoriano (1990-2024): {len(df_ecuador):,}")
print()

# Buscar el terremoto de Pedernales del 16 de abril de 2016
pedernales = df_ecuador[
    (df_ecuador['fecha'].dt.date == pd.Timestamp('2016-04-16').date()) &
    (df_ecuador['magnitud'] >= 7.5)
]
if len(pedernales) == 0:
    print("AVISO: Pedernales 2016 no aparece con esos filtros.")
    print("Usando el sismo de mayor magnitud de 2016 en Ecuador como proxy.")
    pedernales = df_ecuador[df_ecuador['fecha'].dt.year == 2016].nlargest(1, 'magnitud')

evento_ped = pedernales.iloc[0]
alertas_ped= proyectar_alertas(evento_ped)

print("=" * 58)
print("  VALIDACION - TERREMOTO DE PEDERNALES 2016")
print("=" * 58)
print(f"  Lugar           : {evento_ped['lugar']}")
print(f"  Fecha           : {evento_ped['fecha'].date()}")
print(f"  Magnitud        : Mw {evento_ped['magnitud']}")
print(f"  Profundidad     : {evento_ped['profundidad_km']} km")
print(f"  Prob. tsunami   : {evento_ped['prob_tsunami']:.4f}")
print(f"  Tsunami real    : {evento_ped['tsunami']} (USGS catalog flag)")
print("=" * 58)
print()
print("Sistema de alertas por provincia (escenario Pedernales 2016):")
display(alertas_ped)

<br>

## 15.4 Top 10 Sismos de Mayor Riesgo en Ecuador

<br>

In [ ]:
# =============================================================
#  SECCION 15.4 - TOP 10 SISMOS DE MAYOR RIESGO EN ECUADOR
# =============================================================

top10  = df_ecuador.nlargest(10, 'prob_tsunami')
tablas = []

for _, ev in top10.iterrows():
    t = proyectar_alertas(ev)
    t.insert(0, 'Sismo',
             f"{str(ev['fecha'].date())} | M{ev['magnitud']} | {str(ev['lugar'])[:40]}")
    tablas.append(t)

tabla_top10    = pd.concat(tablas, ignore_index=True)
RUTA_ALERTAS   = f"{DIRS['processed']}/alertas_top10_ecuador.csv"
tabla_top10.to_csv(RUTA_ALERTAS, index=False)

print(f"Tabla Top 10 exportada a: {RUTA_ALERTAS}")
print()
display(tabla_top10.head(12))

---

# SECCION 16 - VALIDACION INTERNACIONAL

<br>

Se valida la capacidad de generalizacion del modelo con dos megaterremotos tsunamigenicos documentados fuera de Ecuador: **Tohoku, Japon 2011 (Mw 9.0)** y **Maule, Chile 2010 (Mw 8.8)**.

**Criterio de aceptacion:** `predict_proba > 0.85` en ambos eventos.

<br>

In [ ]:
# =============================================================
#  SECCION 16.1 - VALIDACION TOHOKU 2011 Y MAULE 2010
# =============================================================

tohoku = df_ml[
    (df_ml['fecha'].dt.date == pd.Timestamp('2011-03-11').date()) &
    (df_ml['magnitud'] >= 8.9)
]
maule  = df_ml[
    (df_ml['fecha'].dt.date == pd.Timestamp('2010-02-27').date()) &
    (df_ml['magnitud'] >= 8.7)
]

eventos_int = pd.concat([tohoku.head(1), maule.head(1)]).copy()
eventos_int['Evento']               = ['Tohoku 2011 (Japon)', 'Maule 2010 (Chile)']
eventos_int['Prob. tsunami (modelo)'] = modelo_xgb.predict_proba(eventos_int[FEATURES])[:, 1].round(4)
eventos_int['Tsunami real']          = [
    'SI - tsunami devastador (hasta 40 m en costa de Sendai)',
    'SI - tsunami destructivo (costas del Maule y Biobio)'
]
eventos_int['Cumple umbral 0.85']    = eventos_int['Prob. tsunami (modelo)'] > 0.85

cols_mostrar = ['Evento', 'magnitud', 'profundidad_km',
                'Prob. tsunami (modelo)', 'Tsunami real', 'Cumple umbral 0.85']
display(eventos_int[cols_mostrar].reset_index(drop=True))

<br>

**Analisis critico de la generalizacion:** que el modelo asigne probabilidades altas a Tohoku y Maule indica que aprendio el patron fisico fundamental (magnitud extrema + hipocentro somero + zona de subduccion) sin memorizar particularidades regionales de Ecuador. Limitaciones declaradas: (1) son casos extremos y "faciles" para cualquier umbral razonable; la prueba exigente esta en sismos M 6.5-7.5 donde la frontera es ambigua; (2) ambos eventos estan dentro del periodo de entrenamiento; una validacion estrictamente honesta requeriria un split temporal leave-events-out (propuesto como mejora futura en la Seccion 18).

<br>

---

# SECCION 17 - DASHBOARD INTERACTIVO DE RESULTADOS

<br>

Dashboard construido con **Plotly**: todos los componentes son interactivos (hover con detalle, zoom, pan, leyendas activables). Se exporta como **HTML autonomo** que se puede abrir en cualquier navegador sin Python, ideal para la presentacion final.

| Componente | Contenido |
|---|---|
| **Panel 1** | Curva ROC con AUC anotado |
| **Panel 2** | Feature importance (gain) |
| **Panel 3** | Comparacion AUC-ROC de los 3 modelos |
| **Mapa** | Sismicidad ecuatoriana + alertas por provincia (escenario Pedernales) |
| **Tabla** | Top 10 sismos de mayor riesgo con celdas coloreadas por nivel de alerta |

<br>

## 17.1 Panel Principal

<br>

In [ ]:
# =============================================================
#  SECCION 17.1 - PANEL PRINCIPAL (ROC + Importance + Modelos)
# =============================================================

COLORES_ALERTA = {
    'VERDE'   : '#2ca25f',
    'AMARILLO': '#fec44f',
    'NARANJA' : '#fe9929',
    'ROJO'    : '#d7301f',
}

resumen_top = (tabla_top10
               .sort_values('Score impacto', ascending=False)
               .groupby('Sismo', sort=False).first()
               .reset_index()
               .head(10)[['Sismo', 'Provincia', 'Prob. modelo', 'Score impacto', 'Alerta']])

fig_dash = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'xy'}, {'type': 'xy'}],
           [{'type': 'xy', 'colspan': 2}, None]],
    subplot_titles=(
        'Curva ROC del modelo XGBoost',
        'Importancia de las 7 features',
        'Comparacion AUC-ROC: XGBoost vs Baselines',
    ),
    vertical_spacing=0.17,
)

# Curva ROC interactiva
fig_dash.add_trace(
    go.Scatter(x=fpr, y=tpr, mode='lines',
               name=f'XGBoost (AUC={AUC_XGB:.4f})',
               line=dict(color='#d7301f', width=3),
               hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'),
    row=1, col=1
)
fig_dash.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Azar (AUC=0.50)',
               line=dict(color='gray', dash='dash', width=1.5), showlegend=True),
    row=1, col=1
)

# Feature importance interactiva
imp_orden = importancias.sort_values()
fig_dash.add_trace(
    go.Bar(x=imp_orden.values, y=imp_orden.index, orientation='h',
           marker_color='#2c7fb8', name='Importancia',
           hovertemplate='%{y}: %{x:.4f}<extra></extra>'),
    row=1, col=2
)

# Comparacion de modelos
nombres_m = ['XGBoost', 'Random Forest', 'Reg. Logistica']
aucs_m    = [AUC_XGB,
             roc_auc_score(y_test, proba_rf),
             roc_auc_score(y_test, proba_lr)]
fig_dash.add_trace(
    go.Bar(x=nombres_m, y=aucs_m,
           marker_color=['#d7301f', '#2c7fb8', '#31a354'],
           text=[f'{a:.4f}' for a in aucs_m],
           textposition='outside', name='AUC-ROC', showlegend=False),
    row=2, col=1
)
fig_dash.update_yaxes(range=[0.5, 1.02], row=2, col=1)

fig_dash.update_layout(
    title_text='SeismicPipeline - Dashboard de Resultados del Modelo XGBoost',
    template='plotly_white',
    height=760,
    legend=dict(orientation='h', y=1.06),
)
fig_dash.show()

<br>

## 17.2 Mapa Interactivo de Ecuador

<br>

In [ ]:
# =============================================================
#  SECCION 17.2 - MAPA INTERACTIVO DE ALERTAS POR PROVINCIA
# =============================================================

sub = df_ecuador.sample(min(3000, len(df_ecuador)), random_state=SEED)

fig_mapa = go.Figure()

# Sismicidad historica (coloreada por probabilidad de tsunami)
fig_mapa.add_trace(go.Scattergeo(
    lon=sub['longitud'], lat=sub['latitud'], mode='markers',
    marker=dict(size=4, color=sub['prob_tsunami'], colorscale='YlOrRd',
                cmin=0, cmax=1,
                colorbar=dict(title='Prob. tsunami', x=1.02)),
    name='Sismicidad 1990-2024',
    text=[f"M{m} | prof {d:.0f} km | prob {p:.3f}<br>{str(l)[:50]}"
          for m, d, p, l in zip(sub['magnitud'], sub['profundidad_km'],
                                 sub['prob_tsunami'], sub['lugar'])],
    hovertemplate='%{text}<extra></extra>',
))

# Epicentro de Pedernales 2016
fig_mapa.add_trace(go.Scattergeo(
    lon=[evento_ped['longitud']], lat=[evento_ped['latitud']],
    mode='markers+text',
    marker=dict(size=22, symbol='star', color='black',
                line=dict(color='yellow', width=2)),
    text=[f"Pedernales 2016 (M{evento_ped['magnitud']})"],
    textposition='top center',
    name='Epicentro Pedernales 2016',
    hovertemplate=(f"Pedernales 2016<br>M{evento_ped['magnitud']}"
                   f"<br>Prob: {evento_ped['prob_tsunami']:.3f}<extra></extra>"),
))

# Provincias coloreadas por nivel de alerta
for _, fila in alertas_ped.iterrows():
    plat, plon = PROVINCIAS[fila['Provincia']]
    fig_mapa.add_trace(go.Scattergeo(
        lon=[plon], lat=[plat],
        mode='markers+text',
        marker=dict(size=20, color=COLORES_ALERTA[fila['Alerta']],
                    line=dict(color='black', width=1.5)),
        text=[f"{fila['Provincia']}<br>{fila['Alerta']}"],
        textposition='bottom center',
        textfont=dict(size=10),
        name=f"{fila['Provincia']} - {fila['Alerta']}",
        hovertemplate=(f"{fila['Provincia']}<br>Alerta: {fila['Alerta']}"
                       f"<br>Score: {fila['Score impacto']}"
                       f"<br>Distancia: {fila['Distancia (km)']} km<extra></extra>"),
    ))

fig_mapa.update_geos(
    lataxis_range  =[-6, 3.5],
    lonaxis_range  =[-93.5, -76.5],
    showcountries  =True,  countrycolor='gray',
    showcoastlines =True,  coastlinecolor='gray',
    showland       =True,  landcolor='#f0ede5',
    showocean      =True,  oceancolor='#d6e8f0',
    resolution     =50,
)
fig_mapa.update_layout(
    title='Mapa Interactivo de Alertas por Provincia Costera - Escenario Pedernales 2016',
    template='plotly_white',
    height=620,
    legend=dict(orientation='h', y=-0.05),
)
fig_mapa.show()

<br>

## 17.3 Tabla Interactiva Top 10

<br>

In [ ]:
# =============================================================
#  SECCION 17.3 - TABLA TOP 10 CON COLORES POR NIVEL DE ALERTA
# =============================================================

fig_tabla = go.Figure(data=[go.Table(
    header=dict(
        values     = ['<b>Sismo</b>', '<b>Provincia mas expuesta</b>',
                      '<b>Prob. modelo</b>', '<b>Score impacto</b>', '<b>Alerta</b>'],
        fill_color = '#2c3e50',
        font       = dict(color='white', size=12),
        align      = 'left',
        height     = 35,
    ),
    cells=dict(
        values     = [resumen_top['Sismo'], resumen_top['Provincia'],
                      resumen_top['Prob. modelo'], resumen_top['Score impacto'],
                      resumen_top['Alerta']],
        fill_color = [['white'] * len(resumen_top)] * 4 +
                     [[COLORES_ALERTA[a] for a in resumen_top['Alerta']]],
        align      = 'left',
        height     = 28,
    ),
)])
fig_tabla.update_layout(
    title    = 'Top 10 Sismos de Mayor Riesgo en Ecuador (1990-2024)',
    height   = 430,
    template = 'plotly_white',
)
fig_tabla.show()

In [ ]:
# =============================================================
#  SECCION 17.4 - EXPORTACION DEL DASHBOARD COMO HTML AUTONOMO
# =============================================================

RUTA_DASH = f"{DIRS['processed']}/dashboard_interactivo.html"

with open(RUTA_DASH, 'w', encoding='utf-8') as f:
    f.write('<html><head><meta charset="utf-8">')
    f.write('<title>SeismicPipeline - Dashboard de Resultados</title>')
    f.write('<style>body{font-family:Arial,sans-serif;padding:20px;}'
            'h1{color:#2c3e50}</style></head><body>')
    f.write('<h1>SeismicPipeline - Dashboard de Resultados del Modelo XGBoost</h1>')
    f.write('<p>Universidad Internacional del Ecuador (UIDE) | Proyecto SEIS | 2026</p>')
    f.write('<hr>')
    f.write(fig_dash.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write(fig_mapa.to_html(full_html=False, include_plotlyjs=False))
    f.write(fig_tabla.to_html(full_html=False, include_plotlyjs=False))
    f.write('</body></html>')

print(f"Dashboard interactivo exportado a: {RUTA_DASH}")
print("Se puede abrir en cualquier navegador (Chrome, Firefox, Edge).")
print("Conserva hover, zoom y leyendas sin necesidad de Python.")

---

> **Nota de desarrollo:** Esta seccion corresponde al issue **SEIS-23** (dashboard interactivo).
> - SEIS-96: curva ROC con AUC anotado en grafica interactiva Plotly
> - SEIS-97: feature importance con las 7 variables del modelo
> - SEIS-98: mapa de Ecuador con 6 provincias coloreadas por nivel de alerta
> - SEIS-99: tabla Top 10 sismos de mayor riesgo con niveles de alerta por provincia
> 
> Dashboard exportado como HTML autonomo en `data/processed/dashboard_interactivo.html`.
> Limitaciones, conclusiones y documentacion final continuan en commits siguientes.
